# CLM-8B on the Reuters top-10 benchmark
**Just do two things:** check the GPU says **L4** (top right, or *Runtime → Change runtime type*), then **Runtime → Run all**.

Everything else is automatic: it downloads the code and data, starts CLM, classifies the 2,533 Reuters test stories twice (original questions, then the detailed v3.1 prompt), and downloads a `clm_results.zip` to your computer at the end. Takes roughly 1 hour. Keep this tab open.

In [ ]:
# Step 1 of 5: check the GPU (needs about 24 GB: an L4 or A100).
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'none')
assert gpu and int(gpu.split(',')[-1]) > 20000, ('This GPU is too small for CLM. Choose Runtime -> Change runtime type -> L4 GPU, '
                                                 'then Runtime -> Run all again.')

In [ ]:
# Step 2 of 5: download the code and the Reuters data, install everything (about 5-10 minutes).
%cd /content
!rm -rf fin-decider && git clone -q -b isaac_cwb https://github.com/bansalr/fin-decider.git
%cd /content/fin-decider
!git log --oneline -1
!pip install -q vllm==0.32.0 contrastive-lm==0.1.0 2>&1 | tail -2
!pip install -q -e . 2>&1 | tail -2
!mkdir -p /content/out && pip freeze > /content/out/pip_freeze.txt && nvidia-smi > /content/out/gpu.txt && git rev-parse HEAD > /content/out/commit.txt
!classify prepare -c configs/native_reuters_top10_local.yaml 2>&1 | tail -3

In [ ]:
# Step 3 of 5: start the Qwen3-8B encoder and the CLM server (first start downloads ~16 GB: about 5-15 minutes).
import json, time, requests
def tail(path, n=40):
    print(''.join(open(path).readlines()[-n:]))
def wait(url, proc, name, log, tries=180):
    for _ in range(tries):
        try:
            if requests.get(url, timeout=10).ok:
                print(name, 'is ready'); return
        except requests.RequestException:
            pass
        if proc.poll() is not None:
            tail(log); raise RuntimeError(f'{name} stopped; its log is printed above. Paste it to Claude.')
        time.sleep(10)
    tail(log); raise TimeoutError(f'{name} did not start; its log is printed above. Paste it to Claude.')
encoder = subprocess.Popen(['vllm', 'serve', 'Qwen/Qwen3-8B', '--revision', 'b968826d9c46dd6066d109eabc6255188de91218',
                            '--served-model-name', 'qwen3-8b', '--runner', 'pooling', '--max-model-len', '2048',
                            '--dtype', 'bfloat16', '--port', '8090'],
                           stdout=open('/content/out/vllm.log', 'w'), stderr=subprocess.STDOUT)
wait('http://127.0.0.1:8090/health', encoder, 'Encoder (vLLM)', '/content/out/vllm.log')
server = subprocess.Popen(['clm-serve'], stdout=open('/content/out/clm_server.log', 'w'), stderr=subprocess.STDOUT)
wait('http://127.0.0.1:8700/health', server, 'CLM server', '/content/out/clm_server.log')
probe = {'state': 'Bank of America arranged a 500 million dollar syndicated loan for a utility.',
         'questions': {'q1': {'type': 'noul', 'instructions': 'Does the article describe a bank lending money?'},
                       'q2': {'type': 'noul', 'instructions': 'Is the article about crude oil?',
                              'criteria': {'true': 'Oil prices or production.', 'false': 'Anything else.'}}}}
r = requests.post('http://127.0.0.1:8700/v1/systemone', json=probe, timeout=120)
print('test question:', r.status_code, json.dumps(r.json())[:600])
json.dump({'probe': r.json(), 'health': requests.get('http://127.0.0.1:8700/health').json()},
          open('/content/out/clm_probe.json', 'w'), indent=1)
MODEL_NAME = r.json().get('model')
over = (f'--set models.clm.expected_model_returned={MODEL_NAME}' if MODEL_NAME
        else '--set models.clm.allow_returned_model_mismatch=true')
print('ok' if r.ok else 'PROBLEM: paste this output to Claude', '|', over)

In [ ]:
# Step 4 of 5: classify the 2,533 Reuters test stories, original questions then the detailed v3.1 prompt.
!classify run -c configs/native_reuters_top10_local.yaml -m clm --no-dev-eval --resume {over} 2>&1 | tee /content/out/run_v1.log | tail -25
!classify run -c configs/native_reuters_top10_v3_1_local.yaml -m clm --no-dev-eval --resume {over} 2>&1 | tee /content/out/run_v3_1.log | tail -25

In [ ]:
# Step 5 of 5: bundle the results and download them to your computer.
!cd /content/fin-decider && zip -qr /content/clm_results.zip classification_results_native reports /content/out
from google.colab import files
files.download('/content/clm_results.zip')
print('Done. clm_results.zip is in your Downloads folder: tell Claude.')